# Convert the CSV Data to RDF

This notebook is used to convert the CSV data into RDF format. The CSV files are read and processed, and the resulting RDF data is saved in Turtle format. The code is intentionally very verbose and not optimized for performance, as the main goal is to have a clear and understandable code that can be easily modified and extended.

## Create JSON

In the following part, all information from the different CSV files is combined into a **single deeply nested JSON** structure. This is done by iterating over the rows of the `party_csv` (which is the "main" file) and fetching all the related information from the other connected CSV files.

### Read CSV Files

In [1]:
from pathlib import Path
import pandas as pd

# path to all the CSV files relative to the location of this notebook
CSV_PATH = Path("../data/input")

# main file
party_csv = pd.read_csv(CSV_PATH / "party.csv")

# helper files
party_name_csv = pd.read_csv(CSV_PATH / "party_name.csv")
relationship_csv = pd.read_csv(CSV_PATH / "relationship.csv")
relationship_type_csv = pd.read_csv(CSV_PATH / "relationship_type.csv")
party_code_csv = pd.read_csv(CSV_PATH / "party_code.csv")
party_code_type_csv = pd.read_csv(CSV_PATH / "party_code_type.csv")

In [2]:
display(party_name_csv)

,party_name_id,party_id,notes,sources,translations_languages_code1,translations_abbreviation1,translations_name1,translations_internal_label1,translations_alternative_name1,translations_notes1,...,to_date_datetime,to_date_notes,to_date_sources,translations_languages_code4,translations_abbreviation4,translations_name4,translations_internal_label4,translations_alternative_name4,translations_notes4,translations_sources4
0,1,1,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,SVP,Schweizerische Volkspartei,SVP,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,2,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,SP Schweiz,Sozialdemokratische Partei der Schweiz,SP,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,3,NaN,page_number_string: 5\r\nnotes: \r\nsource_ful...,de-CH,NaN,FDP.Die Liberalen Schweiz,FDPLIB,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,4,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,Mitte,Die Mitte Schweiz,Mitte,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,6,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,GLP,Grünliberale Partei Schweiz,GLP,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
662,672,422,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,DSP der Schweiz,Demokratisch-Soziale Partei der Schweiz,DSP CH,NaN,NaN,...,2012-01-01T12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
663,673,423,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,NaN,Demokratisch-Soziale Partei Baselland,DSP BL,NaN,NaN,...,2010-01-01T12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
664,674,424,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,DSP Graubünden,Demokratisch-Soziale Partei Graubünden,DSP GR,NaN,NaN,...,2017-01-01T12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
665,675,425,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,NaN,Bewegung Öffnung,DSP FR,NaN,NaN,...,2009-01-01T12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Date Handling

All the dates in the CSV and after import to a pandas DataFrame are given as strings in the form of "YYYY-MM-DDT12:00:00" format (always with the time set to 12:00:00). The time part is not relevant for our use case, so we will remove it. Furthermore, the end dates are set to the date of the last valid day, so that schema:validThrough can be used:

- `schema:validFrom` means the first valid day
- `schema:validThrough` means the last valid day

Unknown dates are represented as `NaN` in the DataFrame. To check for unknown dates, we will use the `pd.isna()` function instead of checking for `None`, as the latter does not work with `NaN` values.

In [3]:
def convert_time_column(df, column_name, end_date=False):
    """
    Convert the time column to a date format and remove the time part. 
    If it is an end date, subtract one day to get the last valid day.
    Convert it back to strings in the format YYYY-MM-DD.
    """
    df[column_name] = pd.to_datetime(df[column_name], errors='coerce').dt.date
    if end_date:
        df[column_name] = df[column_name] - pd.Timedelta(days=1)
    df[column_name] = df[column_name].astype(str)
    return df

In [4]:
# convert the time columns to date format and remove the time part

party_csv = convert_time_column(party_csv, 'founding_date_datetime')
party_csv = convert_time_column(party_csv, 'dissolution_date_datetime', end_date=False)
party_name_csv = convert_time_column(party_name_csv, 'from_date_datetime')
party_name_csv = convert_time_column(party_name_csv, 'to_date_datetime', end_date=False)
party_relationship_csv = convert_time_column(relationship_csv, 'from_date_datetime')
party_relationship_csv = convert_time_column(relationship_csv, 'to_date_datetime', end_date=False)

### Create Parties JSON

In [5]:
def parse_list(s):
    """
    Parse a string representation of a list like "[1, 2, 3]" into a Python list.
    """
    if pd.isna(s) or s == "[]":
        return []
    return [int(item) for item in s.strip("[]").split(",")]

In [6]:
def is_currently_valid(day, valid_from, valid_through):
    """
    Check if a given day is within the valid range defined by valid_from and valid_through.
    valid_through can be NaN to indicate that the validity is open-ended. 
    """
    return valid_from <= str(day) and (pd.isna(valid_through) or str(day) <= valid_through)

In [34]:
# create the main dictionary that will hold all the parties and their information
parties = {}

# for each identity (each row in the party_csv)
for _, row in party_csv.iterrows():
    
    # Create a dictionary for the current party
    party = {}
    
    # get the party_id - needed to set the key in the parties dictionary
    party_id = row["party_id"]

    # basic date information
    party["founding_date"] = row["founding_date_datetime"]
    party["dissolution_date"] = row["dissolution_date_datetime"]

    
    #### names ####
    
    # get party names
    party_name_ids = parse_list(row["party_name_id"])

    # all names are stored in a list of dictionaries
    names = []

    for party_name_id in party_name_ids:
        
        # names
        name_lang = {}
        for lang_id in range(1, 5):  # there are 4 language columns (1, 2, 3, 4)
            lang_name = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, f"translations_languages_code{lang_id}"].item()
            name = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, f"translations_name{lang_id}"].item()
            if str(lang_name) != "nan" and str(name) != "nan":
                name_lang[str(lang_name)[0:2]] = name
        
        # abbreviations
        abbr_lang = {}
        for lang_id in range(1, 5):  # there are 4 language columns (1, 2, 3, 4)
            lang_abbr = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, f"translations_languages_code{lang_id}"].item()
            abbr = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, f"translations_abbreviation{lang_id}"].item()
            if str(lang_abbr) != "nan" and str(abbr) != "nan":
                abbr_lang[str(lang_abbr)[0:2]] = abbr
        
        # dates for the name and abbreviation
        valid_from = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, "from_date_datetime"].item()
        valid_through = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, "to_date_datetime"].item()
        
        # create a name entry and add it to the names list
        name_entry = {"id": party_name_id, "name": name_lang, "abbr": abbr_lang, "valid_from": valid_from, "valid_through": valid_through}
        names.append(name_entry)

    
    # sort names by from_date
    names.sort(key=lambda x: x["valid_from"])
    
    # add names to the party dictionary
    party["names"] = names

    
    #### relationships ####

    # get relationships, they are given in two columns, relationship_id_1 and relationship_id_2.
    # They go to either shape-changing or hierarchical relationships.
    relationships = []

    shape_changing_relationship_types = ["successor-of", "split-from", "accessed"]
    hierarchical_relationship_types = ["child-of", "observer-in", "affiliated-with"]

    # get relationships from both columns and combine them into a single list
    relationship_ids = parse_list(row["relationship_id_1"]) + parse_list(row["relationship_id_2"])

    for relationship_id in relationship_ids:
        
        relationship_from = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "party_1_id"].item()
        relationship_to = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "party_2_id"].item()
        relationship_type = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "type"].item()
        relationship_label = relationship_type_csv.loc[relationship_type_csv["relationship_type_id"] == relationship_type, "identifier"].item()
        valid_from = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "from_date_datetime"].item()
        valid_through = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "to_date_datetime"].item()
        
        relationship_entry = {"id": relationship_id, "type_label": relationship_label,"party_from": relationship_from, "party_to": relationship_to, "valid_from": valid_from, "valid_through": valid_through}
        
        if relationship_entry["type_label"] in shape_changing_relationship_types:
            relationship_entry["supra_type_label"] = "shape-changing"
            relationships.append(relationship_entry)
        else:
            # hierarchical relationships are only added on one side because of "version-avalanche" effect.
            if relationship_entry["party_to"] != party_id:
                relationship_entry["supra_type_label"] = "hierarchical"
                relationships.append(relationship_entry)

    # sort relationships by from_date and add them to the party dictionary
    relationships.sort(key=lambda x: x["valid_from"])
    
    # add relationships to the party dictionary
    party["relationships"] = relationships

    
    #### party identifiers ####

    # different identifiers for the same party (e.g. bfs-code, wikidata, etc.)
    party_code_ids = parse_list(row["party_code_id"])
    party_codes = []

    for party_code_id in party_code_ids:
        code = party_code_csv.loc[party_code_csv["party_code_id"] == party_code_id, "code"].item()
        code_type = party_code_csv.loc[party_code_csv["party_code_id"] == party_code_id, "party_code_type_id"].item()
        code_type_label = party_code_type_csv.loc[party_code_type_csv["party_code_type_id"] == code_type, "identifier"].item()

        code_entry = {"id": party_code_id, "code": code, "type_label": code_type_label}
        party_codes.append(code_entry)

    party_codes.append({"code": row["identifier"], "type_label": "ap-identifier"})  # add the identifier from the party.csv row

    #party["party_codes"] = party_codes

    
    # #### versions ####

    # # dictionary to hold the different versions of the party, based on changes in names and relationships over time
    
    # start_date = party["founding_date"]
    
    # versions = {

    #     # first dummy version to start the process, will be overwritten by the first real version
    #     start_date: {
    #         "valid_from": start_date,
    #         "valid_through": None, 
    #         "names": [],
    #         "relationships": [],
    #         "successors": []
    #     }
    # }

    # current_version = start_date

    # helper_dict = {
    #     #"valid_from": start_date,
    #     "names_yesterday": set(),
    #     "relationships_yesterday": set(),
    #     "names_today": set(),
    #     "relationships_today": set()
    # }

    # already_considered_relationships = []

    # date_range = pd.date_range(start=start_date, end="2026-12-31").strftime("%Y-%m-%d").tolist()

    # for i, day in enumerate(date_range):

    #     if day == "1986-06-30" and party_id == 5:
    #         breakpoint()  # Debugging breakpoint for the specific date

    #     for name in names:
    #         # if name is valid on the current day, add it to the current names set
    #         if is_currently_valid(day, name["valid_from"], name["valid_through"]):
    #             helper_dict["names_today"].add(name["id"])

    #     for relationship in relationships:
    #         # if relationship is valid on the current day, add it to the current relationships set
    #         if is_currently_valid(day, relationship["valid_from"], relationship["valid_through"]):
    #             if relationship["supra_type_label"] == "shape-changing":
    #                 # only add shape-changing relationships if they haven't been considered before
    #                 if relationship["id"] not in already_considered_relationships:
    #                     helper_dict["relationships_today"].add(relationship["id"])
    #             else:
    #                 helper_dict["relationships_today"].add(relationship["id"])

    #     # check if the current day is a change point (either a new name or relationship starts or ends)
    #     if (helper_dict["names_today"] != helper_dict["names_yesterday"] or
    #         helper_dict["relationships_today"] != helper_dict["relationships_yesterday"]):

    #         # end current version with the day before the change
    #         versions[current_version]["valid_through"] = date_range[i - 1]  # set the valid_through date for the current version

    #         # add names and relationships from yesterday to the current version (because today they changed and a new version will start)
    #         versions[current_version]["names"] = list(helper_dict["names_yesterday"])
    #         versions[current_version]["relationships"] = list(helper_dict["relationships_yesterday"])

    #         # add successor link
    #         versions[current_version]["successors"].append(f"{party_id}_{day}")  # the next version starts today

    #         # add considered relationships to the already_considered_relationships list to avoid adding them again in future versions
    #         for relationship in relationships:
    #             if relationship["supra_type_label"] == "shape-changing" and relationship["id"] in helper_dict["relationships_yesterday"]:
    #                 already_considered_relationships.append(relationship["id"])
    #                 # remove from the relationships_today set to avoid adding it again in future versions
    #                 helper_dict["relationships_today"].discard(relationship["id"])
            
    #         # start a new version
    #         current_version = day
    #         versions[current_version] = {
    #             "valid_from": current_version,
    #             "valid_through": None,
    #             "names": [],
    #             "relationships": [],
    #             "successors": []
    #         }

    #     # update the "yesterday" sets for the next day
    #     helper_dict["names_yesterday"] = set(helper_dict["names_today"])
    #     helper_dict["relationships_yesterday"] = set(helper_dict["relationships_today"])

    #     # clear the "today" sets for the next day
    #     helper_dict["names_today"].clear()
    #     helper_dict["relationships_today"].clear()

    # # add the last version's names and relationships (because the loop ends usually without a change point)
    # versions[current_version]["names"] = list(helper_dict["names_yesterday"])
    # versions[current_version]["relationships"] = list(helper_dict["relationships_yesterday"])

    # # remove the dummy version if it has no names and relationships (it should not be needed)
    # if not versions[start_date]["names"] and not versions[start_date]["relationships"]:
    #     del versions[start_date]

    # # add versions to the party dictionary
    # party["versions"] = versions


    #### events #### alternative to versions

    events = []

    # add name change events
    for name in names:

        # name start event
        events.append({
            "event_type": "name-change_start",
            "corresponding_id": name["id"],
            "date": name["valid_from"],
        })
        # name stop event if valid_through is not NaN
        if not pd.isna(name["valid_through"]):
            events.append({
                "event_type": "name-change_stop",
                "corresponding_id": name["id"],
                "date": name["valid_through"],
            })

    # add relationship events
    for relationship in relationships:

        # if the relationship is shape-changing, this is an instantaneous event, so we only add one event for it.
        if relationship["supra_type_label"] == "shape-changing":
            events.append({
                "event_type": relationship["type_label"],
                "event_supra_type": relationship["supra_type_label"],
                "corresponding_id": relationship["id"],
                "date": relationship["valid_from"],
                "party_from": relationship["party_from"],
                "party_to": relationship["party_to"]
            })

        # hierarchical relationships are ongoing events, so we add a start and stop event (if applicable) for them.
        else:
            events.append({
                "event_type": f"{relationship["type_label"]}_start",
                "event_supra_type": relationship["supra_type_label"],
                "corresponding_id": relationship["id"],
                "date": relationship["valid_from"],
                "party_from": relationship["party_from"],
                "party_to": relationship["party_to"]
            })
            if not pd.isna(relationship["valid_through"]):
                events.append({
                    "event_type": f"{relationship["type_label"]}_stop",
                    "event_supra_type": relationship["supra_type_label"],
                    "corresponding_id": relationship["id"],
                    "date": relationship["valid_through"],
                    "party_from": relationship["party_from"],
                    "party_to": relationship["party_to"]
                })

    # sort events by date
    events.sort(key=lambda x: x["date"])

    # add founding event at position 0 (first event)
    events.insert(0, {
        "event_type": "founding",
        "date": party["founding_date"],
    })

    # add dissolution event if it exists at last position (last event)
    if not pd.isna(party["dissolution_date"]):
        events.append({
            "event_type": "dissolution",
            "date": party["dissolution_date"],
        })

    # events will only be added to the party dictionary after the mapping 
    # to started and stopped versions is done, because the version_ids are 
    # needed for that mapping. 


    #### event_dates and version_ids #### 

    # list all unique event dates in sorted order, this will be used to create version_ids
    event_dates = sorted(set([event["date"] for event in events]))
    party["event_dates"] = event_dates

    version_ids = event_dates.copy()

    # if the last event is a dissolution event, there is no version after that
    if not pd.isna(party["dissolution_date"]):
        version_ids.remove(party["dissolution_date"])

    party["version_ids"] = version_ids


    #### map events to started or stopped versions ####

    for event in events:

        if event["event_type"] == "founding":
            event["started_version"] = event["date"]
            event["stopped_version"] = None
        if event["event_type"] == "dissolution":
            event["started_version"] = None
            event["stopped_version"] = version_ids[-1]  # the last version before dissolution
        if event["event_type"] == "name-change_start":
            event["started_version"] = event["date"]
            event["stopped_version"] = None
        if event["event_type"] == "name-change_stop":
            event["started_version"] = None
            event["stopped_version"] = event_dates[event_dates.index(event["date"]) - 1] # relevant are the event dates because the event could be on a date where no new version is starting
        if event["event_type"] in ["child-of_start", "observer-in_start", "affiliated-with_start"]:
            event["started_version"] = event["date"]
            event["stopped_version"] = None
        if event["event_type"] in ["child-of_stop", "observer-in_stop", "affiliated-with_stop"]:
            event["started_version"] = None
            event["stopped_version"] = event_dates[event_dates.index(event["date"]) - 1] # relevant are the event dates because the event could be on a date where no new

        if event["event_type"] == "successor-of":
            if event["party_from"] == party_id:
                event["started_version"] = event["date"]
                event["stopped_version"] = None
            else:
                event["started_version"] = None
                event["stopped_version"] = event_dates[event_dates.index(event["date"]) - 1] # relevant are the event dates because the event could be on a date where no new
                event["outside_successor"] = f"{event["party_from"]}_{event["date"]}"  # add the other successor party id to the event

        if event["event_type"] == "split-from":
            if event["party_from"] == party_id:
                event["started_version"] = event["date"]
                event["stopped_version"] = None
            else:
                event["started_version"] = event["date"]
                event["stopped_version"] = event_dates[event_dates.index(event["date"]) - 1] # relevant are the event dates because the event could be on a date where no new
                event["outside_successor"] = f"{event["party_from"]}_{event["date"]}"  # add the other successor party id to the event

        if event["event_type"] == "accessed":
            if event["party_from"] == party_id:
                event["started_version"] = None
                event["stopped_version"] = event_dates[event_dates.index(event["date"]) - 1] # relevant are the event dates because the event could be on a date where no new
                event["outside_successor"] = f"{event["party_from"]}_{event["date"]}"  # add the other successor party id to the event
            else:
                event["started_version"] = event["date"]
                event["stopped_version"] = event_dates[event_dates.index(event["date"]) - 1] # relevant are the event dates because the event could be on a date where no new

    # add events to the party dictionary
    party["events"] = events


    #### versions ####

    versions = {}

    for i, version_id in enumerate(version_ids):
        versions[version_id] = {
            "valid_from": version_id,
            "valid_through": None,
            "is_founding": False,
            "is_dissolution": False,
            "name": None,
            "abbr": None,
            "relationships": [],
            "successors": []
        }

        # if there is a next version, set the valid_through to the next version's date
        if i < len(version_ids) - 1:
            versions[version_id]["valid_through"] = version_ids[i + 1]

            # regular successor within the same party
            versions[version_id]["successors"].append(f"{party_id}_{version_ids[i + 1]}")

        # if it is the last version, one has to check if there is another event 
        # which would terminate the version, e.g. a dissolution event or a name-change_stop event. 
        # If there is no such event, the valid_through is None, meaning the version is still 
        # ongoing.
        else:
            if len(event_dates) > len(version_ids):
                versions[version_id]["valid_through"] = event_dates[-1]

        # check for outside successors (to other parties) in the events
        for event in events:
            if event["stopped_version"] == version_id and "outside_successor" in event:
                versions[version_id]["successors"].append(event["outside_successor"])

        # add the currently valid name
        for name in names:
            if is_currently_valid(version_id, name["valid_from"], name["valid_through"]):
                versions[version_id]["name"] = name["name"]
                versions[version_id]["abbr"] = name["abbr"]
                break  # only one name should be valid at a time

        # add the currently valid hierarchical relationships
        versions[version_id]["relationships"] = []
        hierarchical_relationships = [r for r in relationships if r["supra_type_label"] == "hierarchical"]

        for relationship in hierarchical_relationships:
            if is_currently_valid(version_id, relationship["valid_from"], relationship["valid_through"]):
                versions[version_id]["relationships"].append({
                    "type_label": relationship["type_label"],
                    "party_to": relationship["party_to"]
                })

        # add the starting and stopping events for this version
        versions[version_id]["starting_events"] = [event for event in events if event["started_version"] == version_id]

        # add a flag to indicate if this version is the founding version (if there is a founding event in the starting events)
        if [event for event in versions[version_id]["starting_events"] if event["event_type"] == "founding"]:
            versions[version_id]["is_founding"] = True

        versions[version_id]["stopping_events"] = [event for event in events if event["stopped_version"] == version_id]

        # add a flag to indicate if this version is the dissolution version (if there is a dissolution event in the stopping events)
        if [event for event in versions[version_id]["stopping_events"] if event["event_type"] == "dissolution"]:
            versions[version_id]["is_dissolution"] = True

    # add party versions to the party dictionary
    party["versions"] = versions

    # add party to parties dictionary (finish line operations)
    parties[party_id] = party


In [35]:
import json

print(json.dumps(parties, indent=2, ensure_ascii=False))

{
  "1": {
    "founding_date": "1971-09-22",
    "dissolution_date": NaN,
    "names": [
      {
        "id": 1,
        "name": {
          "de": "Schweizerische Volkspartei",
          "fr": "Union Démocratique du Centre",
          "en": "Schweizerische Volkspartei"
        },
        "abbr": {
          "de": "SVP",
          "fr": "UDC"
        },
        "valid_from": "1971-09-22",
        "valid_through": NaN
      }
    ],
    "relationships": [
      {
        "id": 3,
        "type_label": "successor-of",
        "party_from": 1,
        "party_to": 11,
        "valid_from": "1971-09-22",
        "valid_through": NaN,
        "supra_type_label": "shape-changing"
      },
      {
        "id": 9,
        "type_label": "successor-of",
        "party_from": 1,
        "party_to": 18,
        "valid_from": "1971-09-22",
        "valid_through": NaN,
        "supra_type_label": "shape-changing"
      },
      {
        "id": 5,
        "type_label": "split-from",
        "party_

### Create Versions

In [ ]:
# for all parties and all versions and all relationships
for party_id, party in parties.items():

    # if party_id == 49:
    #     breakpoint()  # Debugging breakpoint for party ID 49

    # create a dictionary to hold the versions for the current party
    versions = {}

    # the version_stream represent the current state of the party and will be updated with each event. It is used to create the versions.
    version_stream = {
        "name": None,
        "split-from_at-beginning": [],
        "split-from_at-end": [],
        "accessed_at-beginning": [],
        "accessed_at-end": [],
        "successor-of_at-beginning": [],
        "successor-of_at-end": [],
        "affiliated-with": [],
        "observer-in": [],
        "child-of": []
    }

    # for each version
    for version_id in party["version_ids"]:

        relevant_events = [event for event in party["events"] if event["date"] == version_id]

        for event in relevant_events:

            if event["event_type"] == "name-change_start":
                version_stream["name"] = event["corresponding_id"]
            elif event["event_type"] == "name-change_stop":
                version_stream["name"] = None
            elif event["event_type"] == "affiliated-with_start":
                version_stream["affiliated-with"].append(event["corresponding_id"])
            elif event["event_type"] == "affiliated-with_stop":
                version_stream["affiliated-with"].remove(event["corresponding_id"])
            elif event["event_type"] == "observer-in_start":
                version_stream["observer-in"].append(event["corresponding_id"])
            elif event["event_type"] == "observer-in_stop":
                version_stream["observer-in"].remove(event["corresponding_id"])
            elif event["event_type"] == "child-of_start":
                version_stream["child-of"].append(event["corresponding_id"])
            elif event["event_type"] == "child-of_stop":
                version_stream["child-of"].remove(event["corresponding_id"])

            elif event["event_type"] in ["successor-of", "split-from", "accessed"]:
                # for shape-changing relationships, we add them to the version_stream as they are instantaneous events
                version_stream[event["event_type"] + "_at-beginning"].append(event["corresponding_id"])

        # create a version dictionary for each version_id
        version = {
            "id": version_id,
            "valid_from": version_id,
            "valid_through": None,
            "name": version_stream["name"],
            "successor-of_at-beginning": list(version_stream["successor-of_at-beginning"]),
            "split-from_at-beginning": list(version_stream["split-from_at-beginning"]),
            "accessed_at-beginning": list(version_stream["accessed_at-beginning"]),
            "affiliated-with": list(version_stream["affiliated-with"]), # list() is used to create a copy of the list, so that changes to version_stream do not affect the versions dictionary
            "observer-in": list(version_stream["observer-in"]),
            "child-of": list(version_stream["child-of"])
        }

        # add version to the versions dictionary
        versions[version_id] = version

        # reset instantaneous events for the next version
        version_stream["successor-of_at-beginning"] = []
        version_stream["split-from_at-beginning"] = []
        version_stream["accessed_at-beginning"] = []

    # add versions to the party dictionary
    party["versions"] = versions

In [29]:
import json

print(json.dumps(parties, indent=2, ensure_ascii=False))

{
  "1": {
    "founding_date": "1971-09-22",
    "dissolution_date": NaN,
    "names": [
      {
        "id": 1,
        "name": {
          "de": "Schweizerische Volkspartei",
          "fr": "Union Démocratique du Centre",
          "en": "Schweizerische Volkspartei"
        },
        "abbr": {
          "de": "SVP",
          "fr": "UDC"
        },
        "valid_from": "1971-09-22",
        "valid_through": NaN
      }
    ],
    "relationships": [
      {
        "id": 3,
        "type_label": "successor-of",
        "party_from": 1,
        "party_to": 11,
        "valid_from": "1971-09-22",
        "valid_through": NaN,
        "supra_type_label": "shape-changing"
      },
      {
        "id": 9,
        "type_label": "successor-of",
        "party_from": 1,
        "party_to": 18,
        "valid_from": "1971-09-22",
        "valid_through": NaN,
        "supra_type_label": "shape-changing"
      },
      {
        "id": 5,
        "type_label": "split-from",
        "party_

In [ ]:
for party_id, party in parties.items():

    versions = {}
    
    # the version_stream represent the current state of the party and will be updated with each event. It is used to create the versions.
    version_stream = {
        "valid_from": party["founding_date"],
        "name": None,
        "affiliated-with": [],
        "observer-in": [],
        "child-of": []
        }

    for event in party["events"]:

        # a new version is needed
        if event["date"] != version_stream["valid_from"]:

            # are there successors
            successors = [e for e in party["events"] if e["event_type"] == "successor-of" and e["date"] == event["date"]]

            if successors:
                version_stream["successor-of_at-beginning"] = [s["corresponding_id"] for s in successors]

            # is this a dissolution event

            # create a new version
            version = {
                "id": version_stream["valid_from"],
                "valid_from": version_stream["valid_from"],
                "valid_through": event["date"],
                "name": version_stream["name"],
                "affiliated-with": list(version_stream["affiliated-with"]), # list() is used to create a copy of the list, so that changes to version_stream do not affect the versions dictionary
                "observer-in": list(version_stream["observer-in"]),
                "child-of": list(version_stream["child-of"])
            }

            # add version to the versions dictionary
            versions[version_stream["valid_from"]] = version

            # update the valid_from for the next version
            version_stream["valid_from"] = event["date"]

        if event["event_type"] == "name-change_start":
            version_stream["name"] = event["corresponding_id"]
        if event["event_type"] == "name-change_stop":
            version_stream["name"] = None

        if event["event_type"] in ["affiliated-with_start", "observer-in_start", "child-of_start"]:
            version_stream[event["event_type"].replace("_start", "")].append(event["corresponding_id"])
            
        if event["event_type"] in ["affiliated-with_stop", "observer-in_stop", "child-of_stop"]:
            try:
                version_stream[event["event_type"].replace("_stop", "")].remove(event["corresponding_id"])
            except ValueError:
                print(f"Warning: Tried to remove {event['corresponding_id']} from {event['event_type'].replace('_stop', '')} but it was not present.")

            
    party["versions"] = versions



## Convert the Deep JSON to RDF

In [ ]:
def get_last_version_before_date(chronology, date):
    last_version = None
    for d in sorted(chronology.keys()):
        if d < date:
            last_version = d
        else:
            break
    return last_version

In [ ]:
from datetime import datetime

def get_all_parent_versions(parent_versions, date_from, date_through, version_from):
    
    # if no date_through is given, use today as the end date
    if pd.isna(date_through):
        date_through = str(datetime.now().strftime("%Y-%m-%d"))
    date_from = str(date_from)[0:10]
    date_through = str(date_through)[0:10]
    
    applicable_parent_versions = []
    
    for entry in parent_versions:
        entry_from = parent_versions[entry]["valid_from"]
        entry_through = parent_versions[entry]["valid_through"]
        if pd.isna(entry_through):
            entry_through = str(datetime.now().strftime("%Y-%m-%d"))
        
        if (entry_from <= date_from and date_from <= entry_through) or (entry_from <= date_through and date_through <= entry_through):
            
            if entry_from <= version_from:
                applicable_parent_versions.append(entry_from)

    return applicable_parent_versions

### URIs

In [ ]:
PARTY_IDENTITY = "https://politics.ld.admin.ch/party-id"
PARTY_VERSION = "https://politics.ld.admin.ch/party-version"
PARTY_VOCAB = "https://politics.schema.ld.admin.ch/party-vocab"
PARTY_PREFIX = "pol"

### Transformation

In [ ]:
from rdflib import Graph, Literal, RDF, URIRef, Namespace

# create graph without binding namespaces to avoid automatic prefixing
g = Graph(bind_namespaces="none")

schema = Namespace("http://schema.org/")
g.bind("schema", schema)
xsd = Namespace("http://www.w3.org/2001/XMLSchema#")
g.bind("xsd", xsd)
vl = Namespace("https://version.link/")
g.bind("vl", vl)
pol = Namespace(PARTY_VOCAB)
g.bind(PARTY_PREFIX, pol)

for party_id, party in parties.items():
    
    # create identity
    identity_uri = URIRef(f"{PARTY_IDENTITY}/{party_id}")
    g.add((identity_uri, RDF.type, vl.Identity))
    if not pd.isna(party["dissolution_date"]):
        g.add((identity_uri, RDF.type, vl.Deprecated))

    
    # create versions
    version_ids = list(party["versions"].keys())

    for i in range(len(version_ids)):
        
        version_id = version_ids[i]
            
        version_uri = URIRef(f"{PARTY_VERSION}/{party_id}_{version_id}")
        g.add((version_uri, RDF.type, vl.Version))
        g.add((version_uri, vl.identity, identity_uri))

        # add dates
        g.add((version_uri, schema.validFrom, Literal(party["versions"][version_id]["valid_from"], datatype=xsd.date)))
        if not pd.isna(party["versions"][version_id]["valid_through"]):
            g.add((version_uri, schema.validThrough, Literal(party["versions"][version_id]["valid_through"], datatype=xsd.date)))

        # add name
        name_entry = party["versions"][version_id]["name"]
        if not pd.isna(name_entry):
            for lang, name in name_entry["name"].items():
                g.add((version_uri, schema.name, Literal(name, lang=lang)))
            for lang, abbr in name_entry["abbr"].items():
                g.add((version_uri, schema.alternateName, Literal(abbr, lang=lang)))

        # link to successors and predecessors
        # only link to successor if the current is not the last versions entry 
        if i < len(version_ids) - 1:
            next_version_id = version_ids[i + 1]
            next_version_uri = URIRef(f"{PARTY_VERSION}/{party_id}_{next_version_id}")
            g.add((version_uri, vl.successor, next_version_uri))
            g.add((next_version_uri, vl.predecessor, version_uri))

        # if it is last entry
        if i == len(version_ids) - 1:
            g.add((identity_uri, vl.version, version_uri))


        # check all relationships for special cases
        for relationship in party["versions"][version_id].get("relationships", []):
            
            # special case split_from, successor_of, accessed are such, that they create new versions for all parties involved
            # special case child-of, observer-in, affiliated-with are more "hierarchical" and do not create new versions on the other side of the relationship (e.g. child-of, observer-in, affiliated-with)

            # special case 1: split-from
            if relationship["type_label"] == "split-from" and relationship["party_from"] == party_id:
                parent_party_id = relationship["party_to"]
                parent_version_id = get_last_version_before_date(parties[parent_party_id]["versions"], version_id)
                if not pd.isna(parent_version_id):
                    parent_version_uri = URIRef(f"{PARTY_VERSION}/{parent_party_id}_{parent_version_id}")
                    g.add((parent_version_uri, vl.successor, version_uri))
                    g.add((parent_version_uri, pol.splitTo, version_uri))
                    g.add((version_uri, vl.predecessor, parent_version_uri))
                    g.add((version_uri, pol.splitFrom, parent_version_uri))

            # special case 2: successor-of
            if relationship["type_label"] == "successor-of" and relationship["party_from"] == party_id:
                parent_party_id = relationship["party_to"]
                parent_version_id = get_last_version_before_date(parties[parent_party_id]["versions"], version_id)
                if not pd.isna(parent_version_id):
                    parent_version_uri = URIRef(f"{PARTY_VERSION}/{parent_party_id}_{parent_version_id}")
                    g.add((parent_version_uri, vl.successor, version_uri))
                    g.add((parent_version_uri, pol.successor, version_uri))
                    g.add((version_uri, vl.predecessor, parent_version_uri))
                    g.add((version_uri, pol.predecessor, parent_version_uri))

            # special case 3: accessed
            if relationship["type_label"] == "accessed" and relationship["party_to"] == party_id:
                parent_party_id = relationship["party_from"]
                parent_version_id = get_last_version_before_date(parties[parent_party_id]["versions"], version_id)
                if not pd.isna(parent_version_id):
                    parent_version_uri = URIRef(f"{PARTY_VERSION}/{parent_party_id}_{parent_version_id}")
                    g.add((parent_version_uri, vl.successor, version_uri))
                    g.add((parent_version_uri, pol.accessed, version_uri))
                    g.add((version_uri, vl.predecessor, parent_version_uri))
                    g.add((version_uri, pol.accessedBy, parent_version_uri))

            # special case 4: child-of
            if relationship["type_label"] == "child-of":
                    parent_party_id = relationship["party_to"]
                    if party_id == 101 and version_id == "1979-03-05":
                        breakpoint()
                    parent_version_ids = get_all_parent_versions(parties[parent_party_id]["versions"], 
                                                                 relationship["valid_from"], 
                                                                 relationship["valid_through"],
                                                                 party["versions"][version_id]["valid_from"])
                    if parent_version_ids:
                        for parent_version_id in parent_version_ids:
                            parent_version_uri = URIRef(f"{PARTY_VERSION}/{parent_party_id}_{parent_version_id}")
                            g.add((version_uri, schema.isPartOf, parent_version_uri))
                            g.add((version_uri, pol.childOf, parent_version_uri))


g.serialize(destination="../data/output/parties.ttl", format="turtle")

<Graph identifier=N244e59a915d645d689ce4bd15750275f (<class 'rdflib.graph.Graph'>)>